# SOXL/SOXX Efficiency Chop Guard v1

Notebook-owned diagnostic for the user's SOXX/SOXL spread and oscillation idea. Earlier direct SOXX actor-feature injection and the simple SOXX relative-weakness cap failed walk-forward validation, so this run tests a narrower sparse execution event: only de-risk when SOXL is already high-positioned and the leveraged ETF shows negative efficiency versus SOXX together with spread-volatility or chop stress.

Governance:
- Baseline: `recovery_reentry_regime_drawdown_guard_v1`
- Direct comparison: `soxx_relative_weakness_walkforward_v1`, `recovery_reentry_2021_trendbreak_cap_v1`
- Closest historical success: fixed-actor execution counterfactuals that improved selected recovery/proxy windows without retraining
- Closest historical failure: direct `balanced_rule_residual_v22_soxx` training and `soxx_relative_weakness_walkforward_v1`, where SOXX signals were absorbed or harmful
- Why this is not a duplicate: this is a regime-gated sparse event requiring LETF efficiency/chop stress, not direct actor feature injection and not plain SOXX weakness threshold tuning
- Role: `diagnose`
- Changed layer: `execution` using factor-derived state
- Target: strict-OOS relative return >= +30pp, strict-OOS MaxDD <= 15% if possible, and no material `wf_2020`, `wf_2022`, or `wf_2023_2024` regression
- Notebook rule: this `.ipynb` is the primary single-run conclusion artifact

External basis:
- Volatility Managed Portfolios: https://www.nber.org/papers/w22208
- Understanding the Tracking Errors of Commodity Leveraged ETFs: https://arxiv.org/abs/1610.09404
- Compounding Effects in Leveraged ETFs: Beyond the Volatility Drag Paradigm: https://arxiv.org/abs/2504.20116
- Qlib GitHub, loose-coupled research-to-execution workflow and RL support: https://github.com/microsoft/qlib
- FinRL GitHub, portfolio-management environments with risk management and constraints: https://github.com/AI4Finance-Foundation/FinRL


In [1]:
from __future__ import annotations

import itertools
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = 'src:.'

from src.soxl_quant.app.budget_counterfactual import compute_metrics, dashboard_daily_frame  # noqa: E402
from src.soxl_quant.data import build_dataset, init_qlib  # noqa: E402
from src.soxl_quant.settings import QLIB_DATA_PATH  # noqa: E402

CARD = 'soxl_soxx_efficiency_chop_guard_v1'
CARD_DIR = ROOT / 'runtime' / 'execution' / 'card' / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)

BASE_CARD = 'recovery_reentry_regime_drawdown_guard_v1'
BASE_DIR = ROOT / 'runtime' / 'execution' / 'card' / BASE_CARD
STRICT_BASE_DAILY = BASE_DIR / f'{BASE_CARD}_soxl_daily.csv'
BASE_JSON = BASE_DIR / f'{BASE_CARD}.json'
BASE_WF_DAILY = {
    key: BASE_DIR / 'walkforward' / key / f'{BASE_CARD}_{key}_soxl_daily.csv'
    for key in ['wf_2020', 'wf_2022', 'wf_2023_2024']
}

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=('soxx_signals', 'soxx_spread_oscillation', 'transition_signals'))
price = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
features = dataset.handler.feature_frame.droplevel('instrument').sort_index()


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def read_daily(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    return frame.sort_values('datetime').reset_index(drop=True)


strict_base = read_daily(STRICT_BASE_DAILY)
base_payload = read_json(BASE_JSON)
base_strict = base_payload['variants'][BASE_CARD]['test_proxy'] if 'variants' in base_payload else base_payload.get('selected_strict', {})
wf_base = {key: read_daily(path) for key, path in BASE_WF_DAILY.items()}
base_wf = {}
for key, frame in wf_base.items():
    json_path = BASE_DIR / 'walkforward' / key / f'{BASE_CARD}_{key}.json'
    payload = read_json(json_path)
    base_wf[key] = payload['variants'][f'{BASE_CARD}_{key}']['test_proxy'] if 'variants' in payload else payload.get('selected', payload.get('selected_strict', {}))

feature_columns = [
    'tracking_error_21',
    'tracking_error_vol_21',
    'decay_63',
    'relative_strength_21',
    'regime_risk',
    'mom_10',
    'mom_21',
    'drawdown_21',
    'vix_percentile_252',
    'below_anchor_ratio_63',
    'anchor_gap_180',
    'SOXL_SOXX_RATIO_Z_252',
    'SOXL_SOXX_RATIO_WIDEN_5',
    'SOXL_SOXX_SPREAD_VOL_Z_21',
    'SOXL_SOXX_CHOP_DIVERGENCE_21',
    'REL_VOLUME_RATIO',
    'PRICE_RATIO_MOM',
    'SOXX_BIAS_10',
    'SIGNED_TE_REVERSION_10',
]
feature_columns = [col for col in feature_columns if col in features.columns]
print('loaded', CARD, 'features', len(feature_columns), 'strict_rows', len(strict_base))


[6116:MainThread](2026-05-24 17:36:14,558) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[6116:MainThread](2026-05-24 17:36:15,213) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[6116:MainThread](2026-05-24 17:36:15,213) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[6116:MainThread](2026-05-24 17:36:17,756) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[6116:MainThread](2026-05-24 17:36:17,758) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[6116:MainThread](2026-05-24 17:36:17,759) INFO - qlib.timer - [log.py:127] - Time cost: 0.005s | Init data Done


loaded soxl_soxx_efficiency_chop_guard_v1 features 19 strict_rows 1605


In [2]:
def feature_join(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[['datetime', 'position']].copy()
    feats = features.reset_index().rename(columns={'index': 'datetime'})
    feats = feats.drop(columns=['instrument'], errors='ignore')
    out = out.merge(feats[['datetime', *feature_columns]], on='datetime', how='left')
    price_df = price.rename('price').reset_index()
    price_df.columns = ['datetime', 'price']
    out = out.merge(price_df, on='datetime', how='inner')
    return out.sort_values('datetime').reset_index(drop=True)


def prepare_context(frame: pd.DataFrame) -> dict:
    enriched = feature_join(frame)
    prices = enriched['price'].to_numpy(dtype=float)
    returns = np.diff(prices) / prices[:-1]
    return {
        'frame': enriched,
        'base_pos': enriched['position'].to_numpy(dtype=float),
        'returns': returns,
        'buyhold_return': float(np.prod(1.0 + returns) - 1.0) * 100.0,
        'values': {col: enriched[col].fillna(0.0).to_numpy(dtype=float) for col in feature_columns},
    }


def fast_metrics_from_position(context: dict, positions: np.ndarray) -> dict:
    frame = context['frame']
    returns = context['returns']
    strat_returns = positions[:-1] * returns
    nav = np.cumprod(1.0 + strat_returns)
    total_return = float(nav[-1] - 1.0) * 100.0
    sharpe = float(np.mean(strat_returns) / np.std(strat_returns) * np.sqrt(252)) if len(strat_returns) > 1 and float(np.std(strat_returns)) > 0.0 else 0.0
    drawdowns = nav / np.maximum.accumulate(nav) - 1.0
    stress_mask = drawdowns < -0.10
    return {
        'rows': int(len(frame)),
        'start': frame['datetime'].iloc[0].date().isoformat(),
        'end': frame['datetime'].iloc[-1].date().isoformat(),
        'return': total_return,
        'sharpe': sharpe,
        'max_dd': abs(float(np.min(drawdowns)) * 100.0),
        'avg_pos': float(np.mean(positions)),
        'stress_avg_pos': float(np.mean(positions[1:][stress_mask])) if np.any(stress_mask) else 0.0,
        'stress_days': int(np.sum(stress_mask)),
        'buy_and_hold_return': context['buyhold_return'],
        'relative_return': total_return - context['buyhold_return'],
        'trades': int(np.sum(np.abs(np.diff(positions)) > 0.001)),
    }


def max_drawdown_event_from_position(context: dict, positions: np.ndarray) -> dict:
    frame = context['frame']
    returns = context['returns']
    nav = np.concatenate([[1.0], np.cumprod(1.0 + positions[:-1] * returns)])
    peak = np.maximum.accumulate(nav)
    dd = nav / peak - 1.0
    trough_idx = int(np.argmin(dd))
    peak_idx = int(np.argmax(nav[: trough_idx + 1]))
    return {
        'peak': frame['datetime'].iloc[peak_idx].date().isoformat(),
        'trough': frame['datetime'].iloc[trough_idx].date().isoformat(),
        'max_dd': abs(float(dd[trough_idx]) * 100.0),
        'peak_position': float(positions[peak_idx]),
        'trough_position': float(positions[trough_idx]),
    }


def rolling_active(raw_gate: np.ndarray, hold_days: int) -> np.ndarray:
    active = np.zeros(len(raw_gate), dtype=bool)
    cooldown = 0
    for idx, gate in enumerate(raw_gate):
        if bool(gate):
            cooldown = max(cooldown, int(hold_days))
        if cooldown > 0:
            active[idx] = True
            cooldown -= 1
    return active


def apply_efficiency_chop_guard_prepared(context: dict, **params) -> tuple[pd.DataFrame, np.ndarray, dict]:
    enriched = context['frame']
    base_pos = context['base_pos']
    values = context['values']
    trend_gate = (
        (base_pos >= params['trend_min_pos'])
        & (values['mom_10'] <= params['trend_max_mom10'])
        & (values['drawdown_21'] <= -params['trend_dd21'])
    )
    efficiency_gate = (
        (base_pos >= params['eff_min_pos'])
        & (values['decay_63'] <= params['max_decay63'])
        & (values['SOXL_SOXX_RATIO_WIDEN_5'] <= params['max_ratio_widen5'])
        & (values['SOXL_SOXX_SPREAD_VOL_Z_21'] >= params['min_spread_vol_z'])
        & (values['relative_strength_21'] <= params['max_rel_strength21'])
    )
    chop_gate = (
        (base_pos >= params['eff_min_pos'])
        & (values['decay_63'] <= params['max_decay63'])
        & (values['SOXL_SOXX_CHOP_DIVERGENCE_21'] >= params['min_chop_div'])
        & (values['relative_strength_21'] <= params['max_rel_strength21'])
    )
    raw_gate = trend_gate | efficiency_gate | chop_gate
    active = rolling_active(raw_gate, int(params['hold_days']))
    guarded_pos = np.where(active, np.minimum(base_pos, params['cap_target']), base_pos)
    out = enriched.copy()
    out['base_position'] = base_pos
    out['position'] = guarded_pos
    out['efficiency_chop_trend_gate'] = trend_gate
    out['efficiency_chop_efficiency_gate'] = efficiency_gate
    out['efficiency_chop_chop_gate'] = chop_gate
    out['efficiency_chop_active'] = active
    out['efficiency_chop_changed'] = np.abs(guarded_pos - base_pos) > 1e-9
    audit = {
        'trend_gate_days': int(trend_gate.sum()),
        'efficiency_gate_days': int(efficiency_gate.sum()),
        'chop_gate_days': int(chop_gate.sum()),
        'active_days': int(active.sum()),
        'changed_days': int(out['efficiency_chop_changed'].sum()),
    }
    return out, guarded_pos, audit


def params_from_row(row: pd.Series) -> dict:
    return {
        'cap_target': float(row['cap_target']),
        'hold_days': int(row['hold_days']),
        'trend_min_pos': float(row['trend_min_pos']),
        'trend_dd21': float(row['trend_dd21']),
        'trend_max_mom10': float(row['trend_max_mom10']),
        'eff_min_pos': float(row['eff_min_pos']),
        'max_decay63': float(row['max_decay63']),
        'max_ratio_widen5': float(row['max_ratio_widen5']),
        'min_spread_vol_z': float(row['min_spread_vol_z']),
        'min_chop_div': float(row['min_chop_div']),
        'max_rel_strength21': float(row['max_rel_strength21']),
    }


STRICT_CTX = prepare_context(strict_base)
WF_CTX = {key: prepare_context(frame) for key, frame in wf_base.items()}


def candidate_row(candidate_id: int, params: dict) -> dict:
    _, positions, audit = apply_efficiency_chop_guard_prepared(STRICT_CTX, **params)
    metric = fast_metrics_from_position(STRICT_CTX, positions)
    dd_event = max_drawdown_event_from_position(STRICT_CTX, positions)
    strict_score = (
        max(metric['max_dd'] - 15.0, 0.0) * 8.0
        + max(30.0 - metric['relative_return'], 0.0) * 3.0
        + max(base_strict['return'] - metric['return'], 0.0) * 0.04
        + max(metric['max_dd'] - base_strict['max_dd'], 0.0) * 20.0
    )
    return {
        'candidate_id': candidate_id,
        **params,
        **metric,
        **audit,
        'dd_improvement_vs_base': base_strict['max_dd'] - metric['max_dd'],
        'return_giveback_vs_base': base_strict['return'] - metric['return'],
        'passes_relative_gate': bool(metric['relative_return'] >= 30.0),
        'passes_15dd_gate': bool(metric['max_dd'] <= 15.0),
        'strict_score': strict_score,
        'maxdd_peak': dd_event['peak'],
        'maxdd_trough': dd_event['trough'],
    }


In [3]:
grid = []
keys = [
    'cap_target',
    'hold_days',
    'trend_min_pos',
    'trend_dd21',
    'trend_max_mom10',
    'eff_min_pos',
    'max_decay63',
    'max_ratio_widen5',
    'min_spread_vol_z',
    'min_chop_div',
    'max_rel_strength21',
]
for values in itertools.product(
    [0.15, 0.25, 0.35, 0.50],
    [5, 10, 15],
    [0.60, 0.70],
    [0.06, 0.08, 0.10],
    [-0.03, -0.05],
    [0.55, 0.65],
    [-0.20, -0.30, -0.40],
    [-0.20, 0.00],
    [0.50, 1.00, 1.50],
    [0.25, 0.75, 1.25],
    [-1.0, 0.0, 1.0],
):
    grid.append(dict(zip(keys, values)))

rows = [candidate_row(i, params) for i, params in enumerate(grid)]
screen = pd.DataFrame(rows).sort_values(
    ['passes_15dd_gate', 'passes_relative_gate', 'strict_score', 'max_dd'],
    ascending=[False, False, True, True],
)
screen_path = CARD_DIR / 'parameter_screen.csv'
screen.to_csv(screen_path, index=False)
eligible = screen[screen['passes_relative_gate'] & (screen['changed_days'] > 0)].copy()
top = eligible.sort_values(['passes_15dd_gate', 'strict_score', 'max_dd'], ascending=[False, True, True]).head(16)
if top.empty:
    top = screen.sort_values(['strict_score', 'max_dd']).head(16)
top_path = CARD_DIR / 'parameter_screen_top.csv'
top.to_csv(top_path, index=False)
print(
    f'parameter_screen rows={len(screen)} target15={int(screen["passes_15dd_gate"].sum())} '
    f'relative_pass={int(screen["passes_relative_gate"].sum())} changed_candidates={int((screen["changed_days"] > 0).sum())}'
)
print(
    top[
        [
            'candidate_id', 'return', 'relative_return', 'sharpe', 'max_dd', 'avg_pos',
            'cap_target', 'hold_days', 'trend_min_pos', 'trend_dd21', 'trend_max_mom10',
            'eff_min_pos', 'max_decay63', 'max_ratio_widen5', 'min_spread_vol_z',
            'min_chop_div', 'max_rel_strength21', 'changed_days', 'trend_gate_days',
            'efficiency_gate_days', 'chop_gate_days', 'active_days', 'strict_score',
            'maxdd_peak', 'maxdd_trough',
        ]
    ].to_string(index=False)
)


parameter_screen rows=46656 target15=0 relative_pass=12360 changed_candidates=46656
 candidate_id      return  relative_return   sharpe    max_dd  avg_pos  cap_target  hold_days  trend_min_pos  trend_dd21  trend_max_mom10  eff_min_pos  max_decay63  max_ratio_widen5  min_spread_vol_z  min_chop_div  max_rel_strength21  changed_days  trend_gate_days  efficiency_gate_days  chop_gate_days  active_days  strict_score maxdd_peak maxdd_trough
        38882 1411.719543       595.207043 1.171697 31.957679 0.413345         0.5         10            0.6        0.06            -0.03         0.55         -0.2              -0.2               0.5          0.25                 1.0           332              140                     4               9          422    170.106771 2024-03-07   2025-04-08
        38885 1411.719543       595.207043 1.171697 31.957679 0.413345         0.5         10            0.6        0.06            -0.03         0.55         -0.2              -0.2               0.5         

In [4]:
wf_rows = []
for _, candidate in top.iterrows():
    candidate_id = int(candidate['candidate_id'])
    params = params_from_row(candidate)
    for wf_key, context in WF_CTX.items():
        _, positions, audit = apply_efficiency_chop_guard_prepared(context, **params)
        metric = fast_metrics_from_position(context, positions)
        base = base_wf[wf_key]
        wf_rows.append({
            'candidate_id': candidate_id,
            'wf': wf_key,
            **params,
            **metric,
            **audit,
            'base_return': base['return'],
            'base_max_dd': base['max_dd'],
            'delta_return_vs_base': metric['return'] - base['return'],
            'delta_max_dd_vs_base': metric['max_dd'] - base['max_dd'],
            'wf_return_penalty': max(base['return'] - metric['return'], 0.0),
            'wf_dd_penalty': max(metric['max_dd'] - base['max_dd'], 0.0),
        })

wf_screen = pd.DataFrame(wf_rows)
wf_path = CARD_DIR / 'walkforward_screen.csv'
wf_screen.to_csv(wf_path, index=False)
wf_summary = wf_screen.groupby('candidate_id').agg(
    wf_return_penalty=('wf_return_penalty', 'sum'),
    wf_dd_penalty=('wf_dd_penalty', 'sum'),
    wf_changed_days=('changed_days', 'sum'),
    min_delta_return_vs_base=('delta_return_vs_base', 'min'),
    max_delta_max_dd_vs_base=('delta_max_dd_vs_base', 'max'),
).reset_index()
wf_2022 = wf_screen[wf_screen['wf'] == 'wf_2022'][['candidate_id', 'delta_return_vs_base', 'delta_max_dd_vs_base']].rename(
    columns={'delta_return_vs_base': 'wf_2022_delta_return', 'delta_max_dd_vs_base': 'wf_2022_delta_max_dd'}
)
wf_summary = wf_summary.merge(wf_2022, on='candidate_id', how='left')
wf_summary['wf_2022_ok'] = (wf_summary['wf_2022_delta_return'] >= -5.0) & (wf_summary['wf_2022_delta_max_dd'] <= 0.75)
selection_table = top.merge(wf_summary, on='candidate_id', how='left')
selection_table['final_score'] = (
    selection_table['strict_score']
    + selection_table['wf_return_penalty'] * 1.2
    + selection_table['wf_dd_penalty'] * 20.0
    + (~selection_table['wf_2022_ok']).astype(float) * 50.0
)
selection_table = selection_table.sort_values(['passes_15dd_gate', 'wf_2022_ok', 'final_score'], ascending=[False, False, True])
selection_table_path = CARD_DIR / 'selection_table.csv'
selection_table.to_csv(selection_table_path, index=False)
wf_summary_path = CARD_DIR / 'walkforward_summary.csv'
wf_summary.to_csv(wf_summary_path, index=False)
wf_screen.head()
selected = selection_table.iloc[0]
selected_params = params_from_row(selected)
print('selected_candidate', int(selected['candidate_id']), json.dumps(selected_params, sort_keys=True))
print(selection_table[['candidate_id','return','relative_return','max_dd','dd_improvement_vs_base','wf_return_penalty','wf_dd_penalty','wf_2022_delta_return','wf_2022_delta_max_dd','wf_2022_ok','final_score']].to_string(index=False))


selected_candidate 39044 {"cap_target": 0.5, "eff_min_pos": 0.65, "hold_days": 10, "max_decay63": -0.2, "max_ratio_widen5": -0.2, "max_rel_strength21": 1.0, "min_chop_div": 0.25, "min_spread_vol_z": 0.5, "trend_dd21": 0.06, "trend_max_mom10": -0.03, "trend_min_pos": 0.6}
 candidate_id      return  relative_return    max_dd  dd_improvement_vs_base  wf_return_penalty  wf_dd_penalty  wf_2022_delta_return  wf_2022_delta_max_dd  wf_2022_ok  final_score
        39044 1411.719543       595.207043 31.957679                0.162501          57.737237       0.329650             -3.885848         -3.552714e-14        True   245.984465
        39071 1411.719543       595.207043 31.957679                0.162501          57.737237       0.329650             -3.885848         -3.552714e-14        True   245.984465
        39047 1411.719543       595.207043 31.957679                0.162501          57.870263       0.329650             -3.885848         -3.552714e-14        True   246.144097
        

In [5]:
final_daily, final_positions, final_audit = apply_efficiency_chop_guard_prepared(STRICT_CTX, **selected_params)
final_metric = fast_metrics_from_position(STRICT_CTX, final_positions)
final_event = max_drawdown_event_from_position(STRICT_CTX, final_positions)
base_event = max_drawdown_event_from_position(STRICT_CTX, STRICT_CTX['base_pos'])
final_payload = {
    'card': CARD,
    'description': 'Notebook-selected SOXL/SOXX leverage-efficiency and chop sparse guard.',
    'baseline_card': BASE_CARD,
    'selected_candidate_id': int(selected['candidate_id']),
    'selected_params': selected_params,
    'baseline_strict': base_strict,
    'selected_strict': final_metric,
    'baseline_drawdown_event': base_event,
    'selected_drawdown_event': final_event,
    'strict_audit': final_audit,
}
json_path = CARD_DIR / f'{CARD}.json'
json_path.write_text(json.dumps(final_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)), encoding='utf-8')
daily_path = CARD_DIR / f'{CARD}_soxl_daily.csv'
dashboard_daily_frame(final_daily[['datetime','position']], price).merge(
    final_daily[[
        'datetime',
        'base_position',
        'efficiency_chop_trend_gate',
        'efficiency_chop_efficiency_gate',
        'efficiency_chop_chop_gate',
        'efficiency_chop_active',
        'efficiency_chop_changed',
        'decay_63',
        'SOXL_SOXX_RATIO_WIDEN_5',
        'SOXL_SOXX_SPREAD_VOL_Z_21',
        'SOXL_SOXX_CHOP_DIVERGENCE_21',
        'relative_strength_21',
    ]],
    on='datetime',
    how='left',
).to_csv(daily_path, index=False)

wf_artifact_rows = []
for wf_key, context in WF_CTX.items():
    guarded, positions, audit = apply_efficiency_chop_guard_prepared(context, **selected_params)
    metric = fast_metrics_from_position(context, positions)
    out_dir = CARD_DIR / 'walkforward' / wf_key
    out_dir.mkdir(parents=True, exist_ok=True)
    wf_json = out_dir / f'{CARD}_{wf_key}.json'
    wf_daily = out_dir / f'{CARD}_{wf_key}_soxl_daily.csv'
    wf_payload = {
        'card': f'{CARD}_{wf_key}',
        'baseline_card': f'{BASE_CARD}_{wf_key}',
        'selected_params': selected_params,
        'baseline': base_wf[wf_key],
        'selected': metric,
        'audit': audit,
    }
    wf_json.write_text(json.dumps(wf_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)), encoding='utf-8')
    dashboard_daily_frame(guarded[['datetime','position']], price).merge(
        guarded[[
            'datetime',
            'base_position',
            'efficiency_chop_trend_gate',
            'efficiency_chop_efficiency_gate',
            'efficiency_chop_chop_gate',
            'efficiency_chop_active',
            'efficiency_chop_changed',
            'decay_63',
            'SOXL_SOXX_RATIO_WIDEN_5',
            'SOXL_SOXX_SPREAD_VOL_Z_21',
            'SOXL_SOXX_CHOP_DIVERGENCE_21',
            'relative_strength_21',
        ]],
        on='datetime',
        how='left',
    ).to_csv(wf_daily, index=False)
    wf_artifact_rows.append({'wf': wf_key, **metric, **audit, 'json_path': str(wf_json.relative_to(ROOT)), 'daily_path': str(wf_daily.relative_to(ROOT))})
final_payload['selected_wf'] = wf_artifact_rows
json_path.write_text(json.dumps(final_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)), encoding='utf-8')
pd.DataFrame(wf_artifact_rows).to_csv(CARD_DIR / 'walkforward_selected_summary.csv', index=False)
print('wrote', json_path.relative_to(ROOT), daily_path.relative_to(ROOT))


wrote runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/soxl_soxx_efficiency_chop_guard_v1.json runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/soxl_soxx_efficiency_chop_guard_v1_soxl_daily.csv


In [6]:
diag_dir = CARD_DIR / 'oos_poor_window_diagnostics'
diag_cmd = [
    sys.executable,
    'scripts/diagnose_oos_poor_windows.py',
    '--daily-csv',
    str(daily_path),
    '--profile',
    CARD,
    '--output-dir',
    str(diag_dir),
    '--oos-start',
    '2020-01-02',
    '--window-size',
    '63',
    '--target-excess',
    '30',
    '--target-max-dd',
    '15',
]
subprocess.run(diag_cmd, cwd=ROOT, check=True)
poor_windows = pd.read_csv(diag_dir / 'poor_windows.csv') if (diag_dir / 'poor_windows.csv').exists() else pd.DataFrame()

wf_lines = []
for row in wf_artifact_rows:
    wf_key = row['wf']
    base = base_wf[wf_key]
    wf_lines.append(
        f"| `{wf_key}` | {base['return']:.2f}% / {base['max_dd']:.2f}% | "
        f"{row['return']:.2f}% / {row['max_dd']:.2f}% | "
        f"{row['return'] - base['return']:.2f}pp / {row['max_dd'] - base['max_dd']:.2f}pp | {int(row['changed_days'])} |"
    )
poor_lines = []
for _, row in poor_windows.head(5).iterrows():
    poor_lines.append(
        f"- {row['start']}..{row['end']}: strategy={row['strategy_return']:.2f}%, "
        f"buyhold={row['buyhold_return']:.2f}%, excess={row['excess_return']:.2f}pp, max_dd={row['max_dd']:.2f}%"
    )
poor_text = '\n'.join(poor_lines) if poor_lines else '- no poor-window rows emitted'

if final_metric['max_dd'] <= 15.0 and final_metric['relative_return'] >= 30.0 and bool(selected['wf_2022_ok']):
    decision = 'Promotion candidate pending broader harness review.'
elif final_metric['max_dd'] < base_strict['max_dd'] - 1.0 and bool(selected['wf_2022_ok']):
    decision = 'Diagnostic keep / no runtime promotion: the sparse SOXL/SOXX efficiency guard reduces strict-OOS drawdown but still misses the 15% target.'
else:
    decision = 'Diagnostic drop / no runtime promotion: the SOXL/SOXX efficiency/chop state does not materially solve strict-OOS drawdown and does not justify retraining.'

readout_lines = [
    '## Final Readout',
    '',
    'This notebook is the primary single-run conclusion artifact.',
    '',
    f"- Selected candidate: `{int(selected['candidate_id'])}`",
    f"- Selected params: `{json.dumps(selected_params, sort_keys=True)}`",
    f"- Strict OOS baseline: `{base_strict['return']:.2f}% / {base_strict['sharpe']:.3f} / {base_strict['max_dd']:.2f}% MaxDD / excess {base_strict['relative_return']:.2f}pp`",
    f"- Strict OOS selected: `{final_metric['return']:.2f}% / {final_metric['sharpe']:.3f} / {final_metric['max_dd']:.2f}% MaxDD / excess {final_metric['relative_return']:.2f}pp`",
    f"- Strict screen target15 candidates: `{int(screen['passes_15dd_gate'].sum())}` / `{len(screen)}`",
    f"- Strict screen relative-pass candidates: `{int(screen['passes_relative_gate'].sum())}` / `{len(screen)}`",
    f"- Baseline MaxDD event: `{base_event['peak']} -> {base_event['trough']} ({base_event['max_dd']:.2f}%)`",
    f"- Selected MaxDD event: `{final_event['peak']} -> {final_event['trough']} ({final_event['max_dd']:.2f}%)`",
    f"- Trend/efficiency/chop/active/changed days: `{final_audit['trend_gate_days']}` / `{final_audit['efficiency_gate_days']}` / `{final_audit['chop_gate_days']}` / `{final_audit['active_days']}` / `{final_audit['changed_days']}`",
    f"- Decision: {decision}",
    '',
    '### Walk-Forward vs Regime Baseline',
    '',
    '| window | baseline return / MaxDD | selected return / MaxDD | delta return / delta MaxDD | changed days |',
    '| --- | ---: | ---: | ---: | ---: |',
    *wf_lines,
    '',
    '### Worst Poor Windows',
    '',
    poor_text,
    '',
    '### Interpretation',
    '',
    '- This test answers the SOXX/SOXL spread question as a sparse execution state, not as direct actor retraining.',
    '- The factor family still does not clear the risk gate: no screened candidate reached the 15% MaxDD target.',
    '- Because the best candidates are dominated by broad trend-break gating while the efficiency/chop sub-gates are sparse, this should not be promoted as a core RL feature or repeated as threshold tuning.',
]
display(Markdown('\n'.join(readout_lines)))


[13497:MainThread](2026-05-24 17:37:05,317) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[13497:MainThread](2026-05-24 17:37:06,051) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[13497:MainThread](2026-05-24 17:37:06,051) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[13497:MainThread](2026-05-24 17:37:08,622) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done
[13497:MainThread](2026-05-24 17:37:08,622) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done
[13497:MainThread](2026-05-24 17:37:08,622) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


oos_diagnostic profile=soxl_soxx_efficiency_chop_guard_v1 windows=12 features=30 oos=2020-01-02..2026-05-21 output=/Users/rziiiii/.openclaw/workspace/runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/oos_poor_window_diagnostics


## Final Readout

This notebook is the primary single-run conclusion artifact.

- Selected candidate: `39044`
- Selected params: `{"cap_target": 0.5, "eff_min_pos": 0.65, "hold_days": 10, "max_decay63": -0.2, "max_ratio_widen5": -0.2, "max_rel_strength21": 1.0, "min_chop_div": 0.25, "min_spread_vol_z": 0.5, "trend_dd21": 0.06, "trend_max_mom10": -0.03, "trend_min_pos": 0.6}`
- Strict OOS baseline: `2272.85% / 1.248 / 32.12% MaxDD / excess 1456.34pp`
- Strict OOS selected: `1411.72% / 1.172 / 31.96% MaxDD / excess 595.21pp`
- Strict screen target15 candidates: `0` / `46656`
- Strict screen relative-pass candidates: `12360` / `46656`
- Baseline MaxDD event: `2021-04-05 -> 2021-05-12 (32.12%)`
- Selected MaxDD event: `2024-03-07 -> 2025-04-08 (31.96%)`
- Trend/efficiency/chop/active/changed days: `140` / `4` / `9` / `422` / `332`
- Decision: Diagnostic drop / no runtime promotion: the SOXL/SOXX efficiency/chop state does not materially solve strict-OOS drawdown and does not justify retraining.

### Walk-Forward vs Regime Baseline

| window | baseline return / MaxDD | selected return / MaxDD | delta return / delta MaxDD | changed days |
| --- | ---: | ---: | ---: | ---: |
| `wf_2020` | 113.89% / 26.14% | 83.91% / 26.14% | -29.98pp / 0.00pp | 47 |
| `wf_2022` | -17.13% / 24.22% | -21.01% / 24.22% | -3.89pp / -0.00pp | 8 |
| `wf_2023_2024` | 58.19% / 30.18% | 34.32% / 30.51% | -23.87pp / 0.33pp | 38 |

### Worst Poor Windows

- 2026-02-05..2026-05-06: strategy=27.99%, buyhold=211.46%, excess=-183.46pp, max_dd=6.42%
- 2026-02-10..2026-05-11: strategy=23.32%, buyhold=200.63%, excess=-177.31pp, max_dd=6.42%
- 2026-02-12..2026-05-13: strategy=22.16%, buyhold=191.98%, excess=-169.82pp, max_dd=6.42%
- 2026-02-13..2026-05-14: strategy=21.18%, buyhold=188.26%, excess=-167.08pp, max_dd=6.42%
- 2025-04-08..2025-07-09: strategy=63.77%, buyhold=223.03%, excess=-159.26pp, max_dd=11.50%

### Interpretation

- This test answers the SOXX/SOXL spread question as a sparse execution state, not as direct actor retraining.
- The factor family still does not clear the risk gate: no screened candidate reached the 15% MaxDD target.
- Because the best candidates are dominated by broad trend-break gating while the efficiency/chop sub-gates are sparse, this should not be promoted as a core RL feature or repeated as threshold tuning.